In [1]:
import joblib
import numpy as np
import pandas as pd

In [2]:
model = joblib.load('/content/model.pkl')
scaler = joblib.load('/content/scaler.pkl')

In [4]:
cols = ['contract_month',
 'security_no',
 'tech_support_no',
 'internet_fiber',
 'payment_check',
 'backup_no',
 'device_protection_no',
 'monthly_charges',
 'paperless_billing',
 'senior_citizen',
 'movies_no',
 'tv_no',
 'tv_yes',
 'movies_yes',
 'multiple_lines_yes',
 'phone_service',
 'gender',
 'multiple_lines_noservice',
 'multiple_lines_no',
 'device_protection_yes',
 'backup_yes',
 'payment_mail',
 'payment_transfer',
 'internet_dsl',
 'payment_creditcard',
 'partner',
 'dependents',
 'tech_support_yes',
 'security_yes',
 'contract_year',
 'total_charges',
 'device_protection_noservice',
 'security_noservice',
 'tech_support_noservice',
 'movies_noservice',
 'tv__no internet service',
 'internet_no',
 'backup_noservice',
 'contract_twoyears',
 'tenure']

In [5]:
def predict_churn_with_cols(customer_data: dict):
    """
    Classifies churn for a single customer using the pre-loaded model and scaler,
    ensuring the input features are ordered according to the global `cols` list.

    Args:
        customer_data (dict): A dictionary where keys are feature names and values are their corresponding data.
                              Example: {'monthly_charges': 50.0, 'tenure': 24, ...}

    Returns:
        int: The predicted churn class (e.g., 0 for no churn, 1 for churn).
    """
    # Create a pandas Series from the input dictionary
    customer_series = pd.Series(customer_data)

    # Reindex the series to match the order of `cols`. Fill missing columns with 0.
    # It's important that `cols` contains all expected features by the model.
    input_features_ordered = customer_series.reindex(cols, fill_value=0)

    # Convert the ordered Series to a 2D numpy array for the scaler and model
    features_array = input_features_ordered.values.reshape(1, -1)

    # Scale the input features
    scaled_features = scaler.transform(features_array)

    # Predict churn using the model
    prediction = model.predict(scaled_features)

    return int(prediction[0])

In [7]:
import gradio as gr

def predict_churn_gradio(
    tenure,
    monthly_charges,
    total_charges,
    gender,
    partner,
    dependents,
    phone_service,
    multiple_lines,
    internet_service,
    online_security,
    online_backup,
    device_protection,
    tech_support,
    streaming_tv,
    streaming_movies,
    contract_type,
    paperless_billing,
    payment_method
):
    customer_data = {col: 0 for col in cols} # Initialize all to 0

    # Numerical features
    customer_data['tenure'] = tenure
    customer_data['monthly_charges'] = monthly_charges
    customer_data['total_charges'] = total_charges

    # Binary/Categorical features mapping

    # Gender (assuming 0 for Male, 1 for Female based on common encoding practices if 'gender' is a single column)
    customer_data['gender'] = 1 if gender == 'Female' else 0

    # Partner
    customer_data['partner'] = 1 if partner == 'Yes' else 0

    # Dependents
    customer_data['dependents'] = 1 if dependents == 'Yes' else 0

    # Phone Service
    customer_data['phone_service'] = 1 if phone_service == 'Yes' else 0

    # Multiple Lines
    if multiple_lines == 'Yes':
        customer_data['multiple_lines_yes'] = 1
    elif multiple_lines == 'No':
        customer_data['multiple_lines_no'] = 1
    elif multiple_lines == 'No phone service': # Corresponds to 'multiple_lines_noservice' in cols
        customer_data['multiple_lines_noservice'] = 1

    # Internet Service
    if internet_service == 'DSL':
        customer_data['internet_dsl'] = 1
    elif internet_service == 'Fiber optic':
        customer_data['internet_fiber'] = 1
    elif internet_service == 'No': # Corresponds to 'internet_no' in cols
        customer_data['internet_no'] = 1

    # Online Security
    if online_security == 'Yes':
        customer_data['security_yes'] = 1
    elif online_security == 'No':
        customer_data['security_no'] = 1
    elif online_security == 'No internet service': # Corresponds to 'security_noservice' in cols
        customer_data['security_noservice'] = 1

    # Online Backup
    if online_backup == 'Yes':
        customer_data['backup_yes'] = 1
    elif online_backup == 'No':
        customer_data['backup_no'] = 1
    elif online_backup == 'No internet service': # Corresponds to 'backup_noservice' in cols
        customer_data['backup_noservice'] = 1

    # Device Protection
    if device_protection == 'Yes':
        customer_data['device_protection_yes'] = 1
    elif device_protection == 'No':
        customer_data['device_protection_no'] = 1
    elif device_protection == 'No internet service': # Corresponds to 'device_protection_noservice' in cols
        customer_data['device_protection_noservice'] = 1

    # Tech Support
    if tech_support == 'Yes':
        customer_data['tech_support_yes'] = 1
    elif tech_support == 'No':
        customer_data['tech_support_no'] = 1
    elif tech_support == 'No internet service': # Corresponds to 'tech_support_noservice' in cols
        customer_data['tech_support_noservice'] = 1

    # Streaming TV
    if streaming_tv == 'Yes':
        customer_data['tv_yes'] = 1
    elif streaming_tv == 'No':
        customer_data['tv_no'] = 1
    elif streaming_tv == 'No internet service': # Corresponds to 'tv__no internet service' in cols
        customer_data['tv__no internet service'] = 1

    # Streaming Movies
    if streaming_movies == 'Yes':
        customer_data['movies_yes'] = 1
    elif streaming_movies == 'No':
        customer_data['movies_no'] = 1
    elif streaming_movies == 'No internet service': # Corresponds to 'movies_noservice' in cols
        customer_data['movies_noservice'] = 1

    # Contract Type
    if contract_type == 'Month-to-month':
        customer_data['contract_month'] = 1
    elif contract_type == 'One year':
        customer_data['contract_year'] = 1
    elif contract_type == 'Two year':
        customer_data['contract_twoyears'] = 1

    # Paperless Billing
    customer_data['paperless_billing'] = 1 if paperless_billing == 'Yes' else 0

    # Senior Citizen
    # If 'senior_citizen' was an explicit Gradio input, it would be mapped here.
    # Since it's not, we'll assume it's set to 0 unless specified otherwise.
    # For simplicity, assuming a default for now if not explicitly passed.
    # If 'senior_citizen' was intended to be an input:
    # customer_data['senior_citizen'] = 1 if senior_citizen == 'Yes' else 0

    # Payment Method
    if payment_method == 'Electronic check':
        customer_data['payment_check'] = 1
    elif payment_method == 'Mailed check':
        customer_data['payment_mail'] = 1
    elif payment_method == 'Bank transfer (automatic)':
        customer_data['payment_transfer'] = 1
    elif payment_method == 'Credit card (automatic)':
        customer_data['payment_creditcard'] = 1

    # Now call the prediction function
    churn_prediction = predict_churn_with_cols(customer_data)

    if churn_prediction == 1:
        return "Customer is likely to Churn." # Assuming 1 means churn
    else:
        return "Customer is likely NOT to Churn." # Assuming 0 means no churn

# Gradio Inputs
inputs = [
    gr.Slider(minimum=0, maximum=72, step=1, label="Tenure (months)", value=12),
    gr.Number(label="Monthly Charges", value=70.0),
    gr.Number(label="Total Charges", value=840.0), # Example: 12 months * 70.0
    gr.Radio(['Male', 'Female'], label='Gender', value='Male'),
    gr.Radio(['Yes', 'No'], label='Partner', value='No'),
    gr.Radio(['Yes', 'No'], label='Dependents', value='No'),
    gr.Radio(['Yes', 'No'], label='Phone Service', value='Yes'),
    gr.Radio(['No phone service', 'No', 'Yes'], label='Multiple Lines', value='No'),
    gr.Radio(['DSL', 'Fiber optic', 'No'], label='Internet Service', value='DSL'),
    gr.Radio(['No internet service', 'No', 'Yes'], label='Online Security', value='No'),
    gr.Radio(['No internet service', 'No', 'Yes'], label='Online Backup', value='No'),
    gr.Radio(['No internet service', 'No', 'Yes'], label='Device Protection', value='No'),
    gr.Radio(['No internet service', 'No', 'Yes'], label='Tech Support', value='No'),
    gr.Radio(['No internet service', 'No', 'Yes'], label='Streaming TV', value='No'),
    gr.Radio(['No internet service', 'No', 'Yes'], label='Streaming Movies', value='No'),
    gr.Radio(['Month-to-month', 'One year', 'Two year'], label='Contract Type', value='Month-to-month'),
    gr.Radio(['Yes', 'No'], label='Paperless Billing', value='Yes'),
    gr.Radio(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], label='Payment Method', value='Electronic check')
]

outputs = gr.Textbox(label="Churn Prediction")

gr.Interface(
    fn=predict_churn_gradio,
    inputs=inputs,
    outputs=outputs,
    title="Customer Churn Prediction",
    description="Predict if a customer will churn based on their service details."
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee501e60ecde9b7bc6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
